## Simple train out and back example

In [1]:
import asyncio
import logging
from pyjmri import Client

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s %(message)s",
    filename="pyjmri.log",
    force=True,
)

## Connect to my layout

In [3]:
jmri = await Client("localhost:12080").__aenter__()
layout = await jmri.discover()
print(f"discovered: {len(layout.turnouts)} turnouts, {len(layout.sensors)} sensors, {len(layout.routes)} routes, {len(layout.blocks)} blocks")

discovered: 52 turnouts, 63 sensors, 15 routes, 38 blocks


## Some helper functions

In [4]:
async def horn(t, *, duration: float = 1.2, pause: float = 0.5, times: int = 3) -> None:
    """Toggle F2 to blow the horn `times` times."""
    for _ in range(times):
        await t.set_function(2, True)
        await asyncio.sleep(duration)
        await t.set_function(2, False)
        await asyncio.sleep(pause)


async def wait_edge(sensor) -> None:
    """Wait for the sensor to go ACTIVE, then back to INACTIVE.

    The trailing edge means the loco has fully passed the detection
    block — used to know the train has cleared a turnout, not just
    reached it.
    """
    print(f"Waiting for {sensor.user_name} to be active")
    await sensor.wait_active()
    print(f"{sensor.user_name} is active")
    print(f"Waiting for {sensor.user_name} to become inactive")
    await sensor.wait_inactive()
    print(f"{sensor.user_name} is now inactive")



async def slow_through(layout, t, schedule: list[tuple[str, float]]) -> None:
    """At each (sensor_user_name, speed) checkpoint, wait for the sensor
    to go ACTIVE, then drop the throttle to the listed speed."""
    for sensor_name, speed in schedule:
        await layout.sensors[sensor_name].wait_active()
        await t.set_speed(speed, forward=True)

In [5]:
async def run_train(layout, dcc, staging_track:str, return_track:str, come_home: asyncio.Event, top_speed:float = 0.2) -> None:
    print(f"[{dcc}] preparing to depart {staging_track}")
    throat = layout.sensors["West / SW"]
    await layout.routes[staging_track].activate()

    try:
        async with layout.throttle(dcc, long=True) as t:
            await t.set_function(0, True)
            await t.set_speed(0.05, forward=True)
            print(f"[{dcc}] departing {staging_track}")
            await t.set_speed(0.2)
            await wait_edge(throat)
            await layout.routes["NW Staging Close"].activate()
            await t.set_speed(top_speed)
            # So now we are running and the mainline has been restored
            print("Now we fire the come home event, and we will fall into the return routine")
            try:
                await come_home.wait()
                print("come_home.wait() returned cleanly")
            except BaseException as e:
                print(f"come_home.wait() raised: {type(e).__name__}: {e}")
                raise
            print(f"{dcc} has been commanded to return to staging")
            await slow_through(layout, t, [
                ("North Zone 9", 0.20),
                ("West / SW", 0.15),
            ])
            await throat.wait_inactive()
            await t.set_speed(0)
            print("Train stopping, ready to reverse")
            await asyncio.sleep(10)
            await layout.routes[return_track].activate()
            # Start the bell for reverse direction
            await t.set_function(1, True)
            await t.set_speed(0.15, forward=False)
            print("Moving in reverse at 0.15")
            print("Moving into the throat")
            await wait_edge(throat)
            print("Throat cleared")
            await t.set_speed(0.10, forward=False)
            print("Moving in reverse for 28 seconds")
            await asyncio.sleep(28)
            print("Stopping")
            await t.set_speed(0)
            await t.set_function(1, False)  #bell off
            await t.set_function(0, False)  #light off
            await layout.routes["NW Staging Close"].activate()  #Restore main line, close NW
            print(f"{dcc} returned and parked in {return_track}")
    except BaseException as e:
        print(f"run_train exiting with: {type(e).__name__}: {e}")
        raise


## Now let's run a train with engine 8997

In [6]:
come_home_8997 = asyncio.Event()
run_8997 = asyncio.create_task(run_train(layout, 8997, "NW Track 6", "NW Track 6", come_home_8997, top_speed=0.4))
print("Train is running. Run the next cell whenever you want it to come home.")

Train is running. Run the next cell whenever you want it to come home.


[8997] preparing to depart NW Track 6
[8997] departing NW Track 6
Waiting for West / SW to be active
West / SW is active
Waiting for West / SW to become inactive
West / SW is now inactive
Now we fire the come home event, and we will fall into the return routine


In [ ]:
come_home_2570 = asyncio.Event()
run_5488 = asyncio.create_task(run_train(layout, 2570, "NW Track 6", "NW Track 6", come_home_2570, top_speed=0.25))
print("Train is running. Run the next cell whenever you want it to come home.")

## Send the come-home signal when you're ready

In [1]:
come_home_8997.set()
print("Come-home signal sent to 8997")

NameError: name 'come_home_8997' is not defined

In [ ]:
come_home_5488.set()
print("Come home signal sent to 5488")

## Wait for the return leg to finish

## Close the client session

In [ ]:
await jmri.__aexit__(None, None, None)
print("CLient closed")

In [ ]:
print("hello")

In [ ]:
  import os, sys
  print("kernel PID:", os.getpid())
  print("python:", sys.executable)
  print()
  for name in ("jmri", "layout", "come_home", "run_8997"):
      print(f"{name!r:<12} in globals: {name in globals()}")
  print()
  # If run_8997 exists:
  try:
      print("run_8997.done():", run_8997.done())
      if run_8997.done():
          print("run_8997.exception():", run_8997.exception())
  except NameError:
      print("run_8997 truly absent")

In [ ]:
print("hello")

In [ ]:
  import os, sys
  print("kernel PID:", os.getpid())
  print("python:", sys.executable)
  print()
  for name in ("jmri", "layout", "come_home", "run_8997"):
      print(f"{name!r:<12} in globals: {name in globals()}")
  print()
  # If run_8997 exists:
  try:
      print("run_8997.done():", run_8997.done())
      if run_8997.done():
          print("run_8997.exception():", run_8997.exception())
  except NameError:
      print("run_8997 truly absent")